# MTHFR AlphaFold Results Analyzer
## One Gene, Seven Disease Pathways, Billions of Lives

**Zero setup required.** Upload your AlphaFold ZIP files and click Run All.

This notebook extracts all confidence metrics, generates PAE heatmaps, creates comparison tables, and produces a complete analysis report.

**GitHub:** [mthfr-gene-therapy-project](https://github.com/DSMPromo/mthfr-gene-therapy-project)

---
*For research and educational purposes only. Not medical advice.*

## Step 1: Upload Your AlphaFold ZIP Files
Run the cell below and use the upload button to select your downloaded ZIP files from AlphaFold Server.

In [ ]:
import os
from google.colab import files

# Create results directory
os.makedirs('results', exist_ok=True)

# Upload ZIP files
print('Upload your AlphaFold Server ZIP files:')
print('(You can select multiple files at once)\n')
uploaded = files.upload()

# Move to results directory
for filename in uploaded:
    os.rename(filename, f'results/{filename}')
    print(f'  Saved: results/{filename}')

print(f'\nTotal files uploaded: {len(uploaded)}')

## Step 2: Extract and Analyze Everything
This cell does all the work automatically.

In [ ]:
import json
import zipfile
import glob
import re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

RESULTS_DIR = Path('results')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

JOB_DESC = {
    '01': 'WT Mono + FAD', '02': 'WT Dimer + FAD',
    '03': 'C677T Mono + FAD', '04': 'C677T Dimer + FAD (TT)',
    '05': 'A1298C Mono + FAD', '06': 'Compound Het Dimer (YOUR GENOTYPE)',
    '07': 'WT Mono (rep)', '08': 'WT Dimer (rep)',
    '09': 'C677T Mono (rep)', '10': 'C677T Dimer (rep)',
    '11': 'A1298C Mono (rep)', '12': 'Compound (rep)',
    '13': 'WT + FAD + THF', '14': 'C677T + FAD + THF',
    '15': 'Compound + FAD + THF', '16': 'WT + FAD + SAM',
}

def extract_job_number(name):
    """Extract 2-digit job number from folder name like 'job01_wt_mono_fad' or 'Job01_WT_mono'."""
    m = re.search(r'(\d{2})', name)
    return m.group(1) if m else '??'

# Unzip all files
print('Extracting ZIP files...')
for zf in RESULTS_DIR.glob('*.zip'):
    extract_dir = RESULTS_DIR / zf.stem
    if not extract_dir.exists():
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(extract_dir)
        print(f'  Extracted: {zf.name}')

# Process each result
all_results = []
print('\nAnalyzing results...')

for result_dir in sorted(RESULTS_DIR.iterdir()):
    if not result_dir.is_dir():
        continue

    summary_files = sorted(result_dir.glob('**/fold_*summary_confidences*.json'))
    full_data_files = sorted(result_dir.glob('**/fold_*full_data*.json'))

    if not summary_files:
        continue

    # Read best model (rank 0)
    with open(summary_files[0]) as f:
        summary = json.load(f)

    result = {
        'name': result_dir.name,
        'ptm': summary.get('ptm'),
        'iptm': summary.get('iptm'),
        'ranking_score': summary.get('ranking_score'),
        'fraction_disordered': summary.get('fraction_disordered'),
        'has_clash': summary.get('has_clash'),
        'chain_ptm': summary.get('chain_ptm', []),
        'chain_iptm': summary.get('chain_iptm', []),
    }

    # Read full data for PAE and pLDDT
    if full_data_files:
        with open(full_data_files[0]) as f:
            full_data = json.load(f)
        result['pae'] = np.array(full_data.get('pae', []))
        result['atom_plddts'] = full_data.get('atom_plddts', [])
        result['mean_plddt'] = np.mean(result['atom_plddts']) if result['atom_plddts'] else None
        result['token_chain_ids'] = full_data.get('token_chain_ids', [])

    all_results.append(result)
    ptm = result['ptm'] or 0
    iptm = result['iptm'] or 0
    mplddt = result.get('mean_plddt') or 0
    jn = extract_job_number(result['name'])
    desc = JOB_DESC.get(jn, result['name'][:25])
    print(f"  {desc}: pTM={ptm:.4f}  ipTM={iptm:.4f}  mean_pLDDT={mplddt:.1f}")

print(f'\nProcessed {len(all_results)} result sets')

## Step 3: Comparison Table
Side-by-side metrics for all jobs.

In [ ]:
# Display comparison table
print(f"{'Job':<30} {'pTM':>8} {'ipTM':>8} {'Score':>10} {'pLDDT':>8} {'Clash':>8}")
print('-' * 82)

for r in all_results:
    jn = extract_job_number(r['name'])
    desc = JOB_DESC.get(jn, r['name'][:28])
    ptm = f"{r['ptm']:.4f}" if r['ptm'] else 'N/A'
    iptm = f"{r['iptm']:.4f}" if r['iptm'] else 'N/A'
    score = f"{r['ranking_score']:.4f}" if r['ranking_score'] else 'N/A'
    plddt = f"{r.get('mean_plddt', 0):.1f}" if r.get('mean_plddt') else 'N/A'
    clash = str(r.get('has_clash', 'N/A'))
    print(f"{desc:<30} {ptm:>8} {iptm:>8} {score:>10} {plddt:>8} {clash:>8}")

## Step 4: PAE Heatmaps
Blue = high confidence. Red = low confidence. Look for changes at the FAD binding interface.

In [ ]:
# Generate PAE heatmaps
n_plots = sum(1 for r in all_results if r.get('pae') is not None and r['pae'].size > 0)

if n_plots == 0:
    print('No PAE data available')
else:
    cols = min(3, n_plots)
    rows = (n_plots + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(7*cols, 6*rows))
    if n_plots == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    idx = 0
    for r in all_results:
        pae = r.get('pae')
        if pae is None or pae.size == 0:
            continue
        ax = axes[idx]
        im = ax.imshow(pae, cmap='bwr_r', vmin=0, vmax=30, aspect='equal')
        jn = extract_job_number(r['name'])
        desc = JOB_DESC.get(jn, r['name'][:25])
        ax.set_title(desc, fontsize=11, fontweight='bold')
        ax.set_xlabel('Token')
        ax.set_ylabel('Token')
        plt.colorbar(im, ax=ax, shrink=0.8)

        # Chain boundaries
        chains = r.get('token_chain_ids', [])
        if chains:
            curr = chains[0]
            for i, c in enumerate(chains):
                if c != curr:
                    ax.axhline(y=i, color='lime', lw=0.5, alpha=0.7)
                    ax.axvline(x=i, color='lime', lw=0.5, alpha=0.7)
                    curr = c
        idx += 1

    # Hide unused subplots
    for i in range(idx, len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout()
    plt.savefig('outputs/pae_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('PAE comparison saved to outputs/pae_comparison.png')

## Step 5: ipTM Comparison Chart
This is the KEY metric — it measures FAD cofactor binding confidence.

In [ ]:
# Bar chart comparing ipTM across all jobs
names = []
iptms = []
colors = []

for r in all_results:
    if r.get('iptm') is None:
        continue
    jn = extract_job_number(r['name'])
    desc = JOB_DESC.get(jn, r['name'][:20])
    names.append(desc)
    iptms.append(r['iptm'])
    # Color code: blue for WT, red for C677T, orange for A1298C, purple for compound
    if 'WT' in desc or 'Wild' in desc:
        colors.append('#2E75B6')
    elif 'Compound' in desc or 'YOUR' in desc:
        colors.append('#6B3FA0')
    elif 'A1298C' in desc:
        colors.append('#D4760A')
    else:
        colors.append('#CC3333')

if iptms:
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(range(len(names)), iptms, color=colors, edgecolor='white')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=10)
    ax.set_xlabel('ipTM Score (higher = more confident FAD binding)', fontsize=12)
    ax.set_title('FAD Binding Confidence: Wild-Type vs Variants', fontsize=14, fontweight='bold')
    ax.axvline(x=0.8, color='gray', linestyle='--', alpha=0.5, label='High confidence threshold')
    ax.legend()

    # Add value labels
    for bar, val in zip(bars, iptms):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10)

    plt.tight_layout()
    plt.savefig('outputs/iptm_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('ipTM comparison saved to outputs/iptm_comparison.png')
else:
    print('No ipTM data available')

## Step 6: Download Results
Download all analysis outputs as a ZIP file.

In [ ]:
import shutil
from google.colab import files

# Save comparison table
with open('outputs/metrics_table.txt', 'w') as f:
    f.write(f"{'Job':<30} {'pTM':>8} {'ipTM':>8} {'Score':>10} {'pLDDT':>8}\n")
    f.write('-' * 74 + '\n')
    for r in all_results:
        jn = extract_job_number(r['name'])
        desc = JOB_DESC.get(jn, r['name'][:28])
        ptm = f"{r['ptm']:.4f}" if r['ptm'] else 'N/A'
        iptm = f"{r['iptm']:.4f}" if r['iptm'] else 'N/A'
        score = f"{r['ranking_score']:.4f}" if r['ranking_score'] else 'N/A'
        plddt = f"{r.get('mean_plddt', 0):.1f}" if r.get('mean_plddt') else 'N/A'
        f.write(f"{desc:<30} {ptm:>8} {iptm:>8} {score:>10} {plddt:>8}\n")

# Zip outputs
shutil.make_archive('MTHFR_Analysis_Results', 'zip', 'outputs')

# Download
print('Downloading analysis results...')
files.download('MTHFR_Analysis_Results.zip')
print('\nDone! Check your downloads folder.')